# Outcome prediction model

Using similar data as went into the thrombolysis probability model - see the train/test split notebook.

This model is used in the thrombolysis decisions app to predict outcomes.

## Code setup

In [25]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import pickle

from dataclasses import dataclass
from xgboost import XGBClassifier

import stroke_utilities.process_data as process_data

In [26]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    data_read_path: str = './stroke_utilities/data/'
    output_folder = './stroke_utilities/output/'
    model_folder = './stroke_utilities/models'
    model_text = 'lgbm_decision_'
    notebook: str = '01_'

paths = Paths()

## Prepare data

In [27]:
train = pd.read_csv(f'{paths.data_read_path}/cohort_10000_train_outcomes.csv')
test = pd.read_csv(f'{paths.data_read_path}/cohort_10000_test_outcomes.csv')

In [28]:
train.columns

Index(['stroke_team_id', 'prior_disability', 'stroke_severity',
       'onset_to_thrombolysis', 'age', 'precise_onset_known',
       'any_afib_diagnosis', 'discharge_disability'],
      dtype='object')

Sanity check:

In [29]:
print(len(train))
print(len(train[train['discharge_disability'] >= 0]))
print(len(train[train['discharge_disability'] <= 6]))

85065
85065
85065


## Run model

In [30]:
# Sample data
# sample = train.sample(frac=1.0, random_state=42+i, replace=True)
sample = train.copy()

In [31]:
X = sample.drop(columns=['discharge_disability'])
y = sample['discharge_disability'].values
y = y.astype(int)

For the XGBoost model, we need to change the single "stroke team ID" column to many individual team columns. For 119 separate teams, we will create 119 new columns. Each column may contain either 1 (meaning "yes") where a patient attended that stroke team, or 0 (meaning "no") where the patient did not attend that stroke team.

In [32]:
X = process_data.one_hot_encode_column(
    X, 'stroke_team_id', prefix='team')

Check that the "stroke_team_id" column has gone and that there are now many "team_" columns.

In [33]:
# Get features
features_ohe = list(X)

# Print the first several...
print(features_ohe[:15])
# ... and last few feature names:
print(features_ohe[-3:])
# The remaining features are all "team_X" for increasing X.

['prior_disability', 'stroke_severity', 'onset_to_thrombolysis', 'age', 'precise_onset_known', 'any_afib_diagnosis', 'team_1', 'team_2', 'team_3', 'team_4', 'team_5', 'team_6', 'team_7', 'team_8', 'team_9']
['team_117', 'team_118', 'team_119']


In [34]:
# Fit full model
model = XGBClassifier(random_state=42)
model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

## Save result

In [35]:
with open(f'{paths.model_folder}/outcome_model.p', 'wb') as fp:
    pickle.dump(model, fp)